## 0. Initialization and Setup

In [1]:
!pip install -q transformers peft datasets bitsandbytes accelerate jsonlines

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 44.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
import os
import json
import jsonlines
from pathlib import Path
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training # quantizing for memory optimization
from datasets import Dataset
import time
import gc

In [4]:
print(f"GPU Available: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

GPU Available: NVIDIA H100 80GB HBM3
Memory: 85.02 GB


In [5]:
# To prevent idle timeout in Colab
import IPython
js_code = '''
function ClickConnect(){
    console.log("Clicking connect button");
    document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)
'''
display(IPython.display.Javascript(js_code))

<IPython.core.display.Javascript object>

## 1. Configuration and Tokenizer Setup

In [6]:
LORA_PATH = Path("/content/drive/MyDrive/OcelotBotV2/lora-datasets/")
OUTPUT_PATH = Path("/content/drive/MyDrive/OcelotBotV2/lora-model-output/")
OUTPUT_PATH.mkdir(exist_ok=True)

In [7]:
# Loading model configuration from data prep
with open(LORA_PATH / "dataset_config.json") as f:
    config = json.load(f)

MAX_LENGTH = config["max_length"]
BASE_MODEL = config["base_model"]

print(f"Base model: {BASE_MODEL}")
print(f"Max length: {MAX_LENGTH} tokens")
print(f"Train samples: {config['train_samples']:,}")
print(f"Val samples: {config['val_samples']:,}")

device = "cuda"

Base model: mistralai/Mistral-7B-v0.1
Max length: 640 tokens
Train samples: 5,676
Val samples: 709


In [8]:
# Tokenizer setup
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [9]:
torch.cuda.empty_cache()
gc.collect()

584

## 2. Dataset Setup

In [10]:
def load_jsonl_dataset(file_path):
    data = []
    with jsonlines.open(file_path) as reader:
        for item in reader:
            full_text = item["prompt"] + " " + item["completion"] + tokenizer.eos_token
            data.append({"text": full_text})

    return Dataset.from_list(data)

train_dataset = load_jsonl_dataset(LORA_PATH / "train.jsonl")
val_dataset = load_jsonl_dataset(LORA_PATH / "val.jsonl")

In [11]:
print(f"Train dataset: {len(train_dataset):,} samples")
print(f"Val dataset: {len(val_dataset):,} samples")

Train dataset: 5,676 samples
Val dataset: 709 samples


In [12]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing train set"
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing val set"
)

Tokenizing train set:   0%|          | 0/5676 [00:00<?, ? examples/s]

Tokenizing val set:   0%|          | 0/709 [00:00<?, ? examples/s]

## 3. Model Setup

In [13]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # Enable 4-bit quantization
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, # Compute in FP16 for speed
    bnb_4bit_use_double_quant=True,       # Double quantization for extra compression
)

# Load with quantization config
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print(f"Parameters: {model.num_parameters():,}")
print(f"Memory: {model.get_memory_footprint() / 1e9:.2f} GB")

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Parameters: 7,241,732,096
Memory: 4.01 GB


In [14]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [15]:
model = prepare_model_for_kbit_training(model) # optimizes memory usage
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
# Enable gradient checkpointing
model.gradient_checkpointing_enable()

trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879


## 4. Training Preparation

In [16]:
per_device_batch_size = 4
gradient_accumulation_steps = 4
effective_batch_size = per_device_batch_size * gradient_accumulation_steps

print(f"Batch configuration:")
print(f"   Per-device batch size: {per_device_batch_size}")
print(f"   Gradient accumulation:  {gradient_accumulation_steps}")
print(f"   Effective batch size:   {effective_batch_size}")

Batch configuration:
   Per-device batch size: 4
   Gradient accumulation:  4
   Effective batch size:   16


In [17]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_PATH),

    # Training schedule
    num_train_epochs=3,
    per_device_train_batch_size=per_device_batch_size,
    per_device_eval_batch_size=per_device_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,

    # Learning rate
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,

    # Optimization
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,

    # Mixed precision for GPU
    fp16=True,

    # Evaluation & saving
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # Logging
    logging_dir=str(OUTPUT_PATH / "logs"),
    logging_steps=50,
    report_to="none",

    gradient_checkpointing=True,

    # Other
    remove_unused_columns=False,
    seed=42
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [18]:
# Data collator for batching inputs
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

## 5. Training and Results

In [19]:
start_time = time.time()
trainer.train()
end_time = time.time()

training_time = end_time - start_time
print(f"Total training time: {training_time/3600:.2f} hours")

Step,Training Loss,Validation Loss
200,1.265986,1.254505
400,1.055265,1.192286
600,1.001306,1.114479
800,0.791185,1.088047
1000,0.728142,1.075518


Total training time: 0.48 hours


In [20]:
final_model_path = OUTPUT_PATH / "final_model"
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"Model saved to: {final_model_path}")

Model saved to: /content/drive/MyDrive/OcelotBotV2/lora-model-output/final_model


In [21]:
# Final metrics
train_loss = None
eval_loss = None

for log in reversed(trainer.state.log_history):
    if train_loss is None and "loss" in log:
        train_loss = log["loss"]
    if eval_loss is None and "eval_loss" in log:
        eval_loss = log["eval_loss"]
    if train_loss is not None and eval_loss is not None:
        break

# Save training metrics
metrics = {
    "training_time_seconds": training_time,
    "training_time_hours": training_time / 3600,
    "final_train_loss": train_loss,
    "final_eval_loss": eval_loss,
    "num_epochs": training_args.num_train_epochs,
    "train_samples": len(train_dataset),
    "val_samples": len(val_dataset),
    "device": "cuda",
    "gpu_name": torch.cuda.get_device_name(0),
    "base_model": BASE_MODEL,
    "max_length": MAX_LENGTH,
    "effective_batch_size": effective_batch_size,
    "learning_rate": training_args.learning_rate,
    "lora_r": lora_config.r,
    "lora_alpha": lora_config.lora_alpha,
    "fp16_enabled": True
}

metrics_file = OUTPUT_PATH / "training_metrics.json"
with open(metrics_file, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"✅ Metrics saved to: {metrics_file}")

✅ Metrics saved to: /content/drive/MyDrive/OcelotBotV2/lora-model-output/training_metrics.json


In [22]:
print(f"\n{'='*70}")
print("TRAINING SUMMARY")
print(f"{'='*70}")
print(f"Final train loss: {train_loss:.4f}" if train_loss else "Final train loss: N/A")
print(f"Final eval loss:  {eval_loss:.4f}" if eval_loss else "Final eval loss: N/A")
print(f"Training time:    {training_time/3600:.2f} hours")
print(f"GPU used:         {torch.cuda.get_device_name(0)}")


TRAINING SUMMARY
Final train loss: 0.7439
Final eval loss:  1.0755
Training time:    0.48 hours
GPU used:         NVIDIA H100 80GB HBM3
